<a href="https://colab.research.google.com/github/veigaeduarda/PAnaM_Webscraping_de_Jornais/blob/main/GAZETA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import ssl # Importação necessária para o contexto não-verificado
from urllib.request import urloppt, Request
from urllib.error import HTTPError
from urllib.parse import quote_plus

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import re


In [ ]:
gazeta = pd.DataFrame()

In [ ]:
eixo_pt_1 = ['política','programa','lei','regulação','restrição','taxação','imposto','tributo','tributação']
eixo_pt_2 = ['política','programa','subsídio','"incpttivo fiscal"','"incpttivos fiscais"','"isptção fiscal"']
eixo_pt_3 = ['política','programa','lei','regulação']
eixo_pt_4 = ['política','programa','lei','regulação','restrição']
eixo_pt_5 = ['política','programa','lei','regulação','restrição']
eixo_pt_6 = ['política','programa','lei','regulação','restrição','imposto','taxação']

assunto_pt_1 = ['ultraprocessado','ultraprocessados','"alimpttos industrializados"','açúcar',
              'ultraprocessada','ultraprocessadas','"bebidas açucaradas"','refrigerante',
              'açúcares','refrigerantes','"bebidas ultraprocessadas"']
assunto_pt_2 = ['"alimpttos saudáveis"','frutas','verduras','"cesta básica"']
assunto_pt_3 = ['"rotulagem de alimpttos"','"rótulo alimpttar"','"rótulo nutricional"','"rótulo frontal"']
assunto_pt_4 = ['"publicidade de alimpttos"','"publicidade de bebidas"','"marketing de alimpttos"','"marketing de bebidas"',
              '"propaganda de alimpttos"','"propaganda de bebidas"']
assunto_pt_5 = ['"alimpttação escolar"','merptda','"cantina escolar"','"lanche escolar"','"refeição escolar"']
assunto_pt_6 = ['pesticidas','agrotóxicos','inseticidas','agroquímicos','"defptsivos agrícolas"']

policies_br = ['"programa de aquisição de alimpttos"','"política nacional de alimpttação e nutrição"',
               '"política nacional de segurança alimpttar e nutricional"','"guia alimpttar para a população brasileira"',
               '"programa nacional de alimpttação escolar"','"programa nacional de fortalecimptto da agricultura familiar"',
               'pronaf']

In [ ]:
queries_pt = []
eixo = [eixo_pt_1,eixo_pt_2,eixo_pt_3,eixo_pt_4,eixo_pt_5, eixo_pt_6]
assunto = [assunto_pt_1,assunto_pt_2,assunto_pt_3,assunto_pt_4,assunto_pt_5,assunto_pt_6]

for i in list(range(1,7)):
  lc = eixo[i-1] # Adjusted index to be 0-based
  lt = assunto[i-1] # Adjusted index to be 0-based
  for j in lc:
    for k in lt:
      # junta as palavras-chave
      query = j+" "+k
      #troca espaços por + e aspectos pelo código que o site entende
      #query = query.replace(' ','+').replace('ç','%C3%A7').replace('ã','%C3%A3').replace('í','%C3%AD').replace('ú','%25C3%25BA').replace('õ','%25C3%25B5').replace("ó","%25C3%25B3").replace('á','%25C3%25A1')
      queries_pt.append(query)

print(queries_pt)
print(len(queries_pt))

['política ultraprocessado', 'política ultraprocessados', 'política "alimpttos industrializados"', 'política açúcar', 'política ultraprocessada', 'política ultraprocessadas', 'política "bebidas açucaradas"', 'política refrigerante', 'política açúcares', 'política refrigerantes', 'política "bebidas ultraprocessadas"', 'programa ultraprocessado', 'programa ultraprocessados', 'programa "alimpttos industrializados"', 'programa açúcar', 'programa ultraprocessada', 'programa ultraprocessadas', 'programa "bebidas açucaradas"', 'programa refrigerante', 'programa açúcares', 'programa refrigerantes', 'programa "bebidas ultraprocessadas"', 'lei ultraprocessado', 'lei ultraprocessados', 'lei "alimpttos industrializados"', 'lei açúcar', 'lei ultraprocessada', 'lei ultraprocessadas', 'lei "bebidas açucaradas"', 'lei refrigerante', 'lei açúcares', 'lei refrigerantes', 'lei "bebidas ultraprocessadas"', 'regulação ultraprocessado', 'regulação ultraprocessados', 'regulação "alimpttos industrializados"',

In [ ]:


queries = queries_pt

gazeta_base = "https://www.gazetadopovo.com.br/busca/?q="

# Initialize sets and lists for efficiptt collection and uniquptess
urls_set = set()
urls_list = []
site_list = []
busca_list = []

# User-Agptt header to mimic a browser and avoid 403 Forbiddpt errors
headers = {"User-Agptt": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"}

# Aggressive filtering patterns to exclude irrelevant URLs early
exclude_patterns = [
    r'^#',  # Anchors
    r'^//', # Protocol-relative links
    r'login\.gazeta\.com\.br',
    r'assinaturas\.gazeta\.com\.br',
    r'conta\.gazetadopovo\.com\.br',
    r'atptdimptto\.gazeta\.com\.br',
    r'x\.com/gazetadopovo',
    r'www\.instagram\.com/gazetadopovo',
    r'politica-de-priva',
    r'pt-br\.facebook\.com/gazetadopovo',
    r'www\.youtube\.com/gazetadopovo',
    r'www\.whatsapp\.com/channel/',
    r'www\.gazetadopovo\.com\.br/redes-sociais/',
    r'www\.gazetadopovo\.com\.br/sobre/',
    r'www\.gazetadopovo\.com\.br/expediptte/',
    r'www\.gazetadopovo\.com\.br/mapa/',
    r'www\.gazetadopovo\.com\.br/termos-de-uso/',
    r'especiais\.gazetadopovo\.com\.br/about-gazeta-do-povo/',
    r'vozes', # Exclude "Vozes" section
    r'opiniao', # Exclude opinion pieces
    r'videos', # Exclude video pages
    r'saber', # Exclude "Saber" section
    r'ultimas-noticias' # Exclude gpteral latest news page
]

combined_pattern = '|'.join(exclude_patterns)

# Loop through queries and pages to collect URLs
for query in queries:
    for n in range(1, 310, 25):
        ptcoded_query = quote_plus(query)
        url_search_page = gazeta_base + ptcoded_query + "&pagina=" + str(n)
        print(f"Buscando na página: {url_search_page}")

        try:
            response = requests.get(url_search_page, headers=headers, timeout=10)
            response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
            soup = BeautifulSoup(response.text, 'html.parser')

            for link in soup.find_all('a', href=True):
                url_candidate = link['href']

                # Step 5: Implemptt aggressive initial filtering
                # 1. ptsure it's an absolute URL
                if not (url_candidate.startswith('http://') or url_candidate.startswith('https://')):
                    continue

                # 2. ptsure it's from gazetadopovo.com.br and not a search result page itself
                if "gazetadopovo.com.br" not in url_candidate or "/busca/?q=" in url_candidate:
                    continue

                # 3. Apply aggressive exclusion patterns
                if re.search(combined_pattern, url_candidate, re.IGNORECASE):
                    continue

                # Step 4: Collect unique URLs, site, and busca values
                if url_candidate not in urls_set:
                    urls_set.add(url_candidate)
                    urls_list.append(url_candidate)
                    site_list.append("Gazeta")
                    busca_list.append(query)

        except requests.exceptions.RequestException as e:
            print(f"Erro na requisição para {url_search_page}: {e}")
            continue # Skip to the next URL on any request error

# Step 6 & 7: Create the DataFrame once after all loops and replace the original 'gazeta'
gazeta = pd.DataFrame({
    'Urls': urls_list,
    'Sites': site_list,
    'Buscas': busca_list
})

# Step 8: Print the head of the new gazeta DataFrame to verify
print("\nRaspagem inicial de URLs concluída. Primeiras linhas do DataFrame 'gazeta':")
display(gazeta.head())
print(f"Total de URLs coletadas: {len(gazeta)}")

Buscando na página: https://www.gazetadopovo.com.br/busca/?q=pol%C3%ADtica+ultraprocessado&pagina=1
Buscando na página: https://www.gazetadopovo.com.br/busca/?q=pol%C3%ADtica+ultraprocessado&pagina=26
Buscando na página: https://www.gazetadopovo.com.br/busca/?q=pol%C3%ADtica+ultraprocessado&pagina=51
Buscando na página: https://www.gazetadopovo.com.br/busca/?q=pol%C3%ADtica+ultraprocessado&pagina=76
Buscando na página: https://www.gazetadopovo.com.br/busca/?q=pol%C3%ADtica+ultraprocessado&pagina=101
Buscando na página: https://www.gazetadopovo.com.br/busca/?q=pol%C3%ADtica+ultraprocessado&pagina=126
Buscando na página: https://www.gazetadopovo.com.br/busca/?q=pol%C3%ADtica+ultraprocessado&pagina=151
Buscando na página: https://www.gazetadopovo.com.br/busca/?q=pol%C3%ADtica+ultraprocessado&pagina=176
Buscando na página: https://www.gazetadopovo.com.br/busca/?q=pol%C3%ADtica+ultraprocessado&pagina=201
Buscando na página: https://www.gazetadopovo.com.br/busca/?q=pol%C3%ADtica+ultraprocess

,Urls,Sites,Buscas
0,https://bsky.app/profile/gazetadopovo.com.br,Gazeta,política ultraprocessado
1,https://assinaturas.gazetadopovo.com.br/por-qu...,Gazeta,política ultraprocessado
2,https://www.gazetadopovo.com.br/expediente/?re...,Gazeta,política ultraprocessado
3,https://www.gazetadopovo.com.br/republica/?ref...,Gazeta,política ultraprocessado
4,https://www.gazetadopovo.com.br/parana/?ref=fo...,Gazeta,política ultraprocessado


Total de URLs coletadas: 1013


In [ ]:
display(gazeta.head())

,Urls,Sites,Buscas
0,https://bsky.app/profile/gazetadopovo.com.br,Gazeta,política ultraprocessado
1,https://assinaturas.gazetadopovo.com.br/por-qu...,Gazeta,política ultraprocessado
2,https://www.gazetadopovo.com.br/republica/?ref...,Gazeta,política ultraprocessado
3,https://www.gazetadopovo.com.br/parana/?ref=fo...,Gazeta,política ultraprocessado
4,https://www.gazetadopovo.com.br/mundo/?ref=footer,Gazeta,política ultraprocessado




---



---



Quantas matérias a raspagpts trouxe?

In [ ]:
gazeta.head ()

,Urls,Sites,Buscas
0,https://bsky.app/profile/gazetadopovo.com.br,Gazeta,política ultraprocessado
1,https://assinaturas.gazetadopovo.com.br/por-qu...,Gazeta,política ultraprocessado
2,https://www.gazetadopovo.com.br/expediente/?re...,Gazeta,política ultraprocessado
3,https://www.gazetadopovo.com.br/republica/?ref...,Gazeta,política ultraprocessado
4,https://www.gazetadopovo.com.br/parana/?ref=fo...,Gazeta,política ultraprocessado


In [ ]:
print(len(gazeta))

1013


In [ ]:
print(f"Número de matérias únicas no DataFrame 'tudo': {len(gazeta)}")

Número de matérias únicas no DataFrame 'tudo': 1013


In [ ]:
# Cria um novo dataframe, tirando duplicatas
tudo = gazeta.drop_duplicates(subset=['Urls'], keep='first').reset_index(drop=True)

# Define padrões para URLs a serem excluídas (links que não são artigos de notícias)
exclude_patterns = [
    r'^#',  # Âncoras (ex: #conteudo)
    r'^//', # Links relativos ao protocolo (ex: //assinaturas.gazeta.com.br)
    r'login\\.gazeta\\.com\\.br',
    r'assinaturas\\.gazeta\\.com\\.br',
    r'conta.gazetadopovo.com.br', #assinatura
    r'atptdimptto\\.gazeta\\.com\\.br',
    r'x\\.com/gazetadopovo',
    r'www\\.instagram\\.com/gazetadopovo',
    r'politica-de-priva', # Política de privacidade
    r'search\\.gazetadopovo\\.com\\.br', # Páginas de resultados de busca
    r'pt-br\\.facebook\\.com/gazetadopovo',
    r'www\\.youtube\\.com/gazetadopovo',
    r'www\\.whatsapp\\.com/channel/',
    r'www\\.gazetadopovo\\.com\\.br/redes-sociais/',
    r'www\\.gazetadopovo\\.com\\.br/sobre/',
    r'www\\.gazetadopovo\\.com\\.br/expediptte/',
    r'www\\.gazetadopovo\\.com\\.br/mapa/',
    r'www\\.gazetadopovo\\.com\\.br/termos-de-uso/',
    r'especiais\\.gazetadopovo\\.com\\.br/about-gazeta-do-povo/'
]

# Combina os padrões em uma única expressão regular
combined_pattern = '|'.join(exclude_patterns)

# Filtra o DataFrame para manter apptas as URLs que NÃO correspondem a npthum dos padrões de exclusão
initial_count = len(tudo)
cleaned_tudo = tudo[~tudo['Urls'].str.contains(combined_pattern, regex=True, na=False)].copy()

# Garante que a URL é uma URL absoluta (começa com http ou https)
cleaned_tudo = cleaned_tudo[cleaned_tudo['Urls'].str.contains(r'^https?://', regex=True, na=False)]

# Remove duplicatas novamptte para garantir, caso o filtro anterior não tptha pego todas
cleaned_tudo.drop_duplicates(subset=['Urls'], keep='first', inplace=True)

tudo = cleaned_tudo.reset_index(drop=True)

# Exibe algumas estatísticas da limpeza
print(f"Número de URLs antes da limpeza adicional: {initial_count}")
print(f"Número de URLs após a limpeza adicional: {len(tudo)}")

# Exibe as primeiras linhas do DataFrame limpo
display(tudo.head())

Número de URLs antes da limpeza adicional: 1013
Número de URLs após a limpeza adicional: 1013


,Urls,Sites,Buscas
0,https://bsky.app/profile/gazetadopovo.com.br,Gazeta,política ultraprocessado
1,https://assinaturas.gazetadopovo.com.br/por-qu...,Gazeta,política ultraprocessado
2,https://www.gazetadopovo.com.br/expediente/?re...,Gazeta,política ultraprocessado
3,https://www.gazetadopovo.com.br/republica/?ref...,Gazeta,política ultraprocessado
4,https://www.gazetadopovo.com.br/parana/?ref=fo...,Gazeta,política ultraprocessado


In [ ]:
tudo['Buscas'] = tudo['Buscas'].str.replace("'",'').str.replace('+',' ').str.replace('%C3%A7','ç').str.replace('%C3%A3','ã').str.replace('%C3%AD','í')
tudo.head()

,Urls,Sites,Buscas
0,https://bsky.app/profile/gazetadopovo.com.br,Gazeta,política ultraprocessado
1,https://assinaturas.gazetadopovo.com.br/por-qu...,Gazeta,política ultraprocessado
2,https://www.gazetadopovo.com.br/expediente/?re...,Gazeta,política ultraprocessado
3,https://www.gazetadopovo.com.br/republica/?ref...,Gazeta,política ultraprocessado
4,https://www.gazetadopovo.com.br/parana/?ref=fo...,Gazeta,política ultraprocessado


In [ ]:
pd.pivot_table(tudo,index='Buscas',values='Urls',aggfunc=len).sort_values(by='Urls',ascending=False)

,Urls
Buscas,
política frutas,24
"subsídio ""cesta básica""",24
"programa ""publicidade de bebidas""",22
programa inseticidas,22
política açúcar,21
...,...
restrição ultraprocessados,1
taxação agrotóxicos,1
subsídio verduras,1


In [ ]:
tudo.head (1000)

,Urls,Sites,Buscas
0,https://bsky.app/profile/gazetadopovo.com.br,Gazeta,política ultraprocessado
1,https://assinaturas.gazetadopovo.com.br/por-qu...,Gazeta,política ultraprocessado
2,https://www.gazetadopovo.com.br/expediente/?re...,Gazeta,política ultraprocessado
3,https://www.gazetadopovo.com.br/republica/?ref...,Gazeta,política ultraprocessado
4,https://www.gazetadopovo.com.br/parana/?ref=fo...,Gazeta,política ultraprocessado
...,...,...,...
995,https://www.gazetadopovo.com.br/economia/econo...,Gazeta,imposto agrotóxicos
996,https://www.gazetadopovo.com.br/republica/refo...,Gazeta,imposto agrotóxicos
997,https://www.gazetadopovo.com.br/agronegocio/di...,Gazeta,imposto agrotóxicos
998,https://www.gazetadopovo.com.br/republica/brev...,Gazeta,imposto agrotóxicos


In [ ]:
# ==========================================
# PEGAR DATA E AUTOR DAS URLS
# ==========================================

import requests
from bs4 import BeautifulSoup
import time

headers = {
    "User-Agptt": "Mozilla/5.0"
}

# cria as colunas caso não existam
if "Data" not in tudo.columns:
    tudo["Data"] = ""

if "Autor" not in tudo.columns:
    tudo["Autor"] = ""

# loop nas urls
for i, row in tudo.iterrows():

    url = row["Urls"]

    try:

        print(f"Acessando {i}: {url}")

        response = requests.get(
            url,
            headers=headers,
            timeout=10
        )

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        # ==========================================
        # AUTOR
        # ==========================================

        autor = "Não obtido"

        autor_meta = soup.find(
            "meta",
            attrs={"name": "author"}
        )

        if autor_meta:
            autor = autor_meta.get("content")

        # fallback
        if autor == "Não obtido":

            autor_meta = soup.find(
                "meta",
                attrs={"property": "author"}
            )

            if autor_meta:
                autor = autor_meta.get("content")

        # ==========================================
        # DATA
        # ==========================================

        data = "Não obtida"

        data_meta = soup.find(
            "meta",
            attrs={
                "property": "article:published_time"
            }
        )

        if data_meta:
            data = data_meta.get("content")

        # fallback
        if data == "Não obtida":

            time_tag = soup.find("time")

            if time_tag:

                data = (
                    time_tag.get("datetime")
                    or
                    time_tag.get_text(strip=True)
                )

        # salva no dataframe
        tudo.loc[i, "Autor"] = autor
        tudo.loc[i, "Data"] = data

        print(f"{i} OK")

        time.sleep(1)

    except Exception as e:

        print(f"Erro na linha {i}: {e}")

        tudo.loc[i, "Autor"] = "Erro"
        tudo.loc[i, "Data"] = "Erro"

# ==========================================
# SALVAR CSV FINAL
# ==========================================

tudo.to_csv(
    "resultado_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Finalizado.")

Acessando 0: https://bsky.app/profile/gazetadopovo.com.br
0 OK
Acessando 1: https://assinaturas.gazetadopovo.com.br/por-que-assinar/?ref=footer
1 OK
Acessando 2: https://www.gazetadopovo.com.br/expediente/?ref=footer
2 OK
Acessando 3: https://www.gazetadopovo.com.br/republica/?ref=footer
3 OK
Acessando 4: https://www.gazetadopovo.com.br/parana/?ref=footer
4 OK
Acessando 5: https://www.gazetadopovo.com.br/mundo/?ref=footer
5 OK
Acessando 6: https://www.gazetadopovo.com.br/economia/?ref=footer
6 OK
Acessando 7: https://www.gazetadopovo.com.br/vida-e-cidadania/?ref=footer
7 OK
Acessando 8: https://www.gazetadopovo.com.br/educacao/?ref=footer
8 OK
Acessando 9: https://www.gazetadopovo.com.br/ideias/?ref=footer
9 OK
Acessando 10: https://www.gazetadopovo.com.br/bomgourmet/?ref=footer
10 OK
Acessando 11: https://www.gazetadopovo.com.br/haus/?ref=footer
Erro na linha 11: HTTPSConnectionPool(host='revistahaus.com.br', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("

In [ ]:
import re
import requests
from bs4 import BeautifulSoup
import json
import time
import pandas as pd

def extract_metadata_v3(url):
    headers = {"User-Agptt": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"}
    res = {"Data": "Não obtida", "Editoria": "Não obtida", "Autor": "Não obtido"}

    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200: return res
        soup = BeautifulSoup(response.text, 'html.parser')

        # 1. JSON-LD
        scripts = soup.find_all('script', type='application/ld+json')
        for script in scripts:
            try:
                data = json.loads(script.string)
                items = data.get('@graph', [data]) if isinstance(data, dict) else []
                for item in items:
                    if isinstance(item, dict):
                        if res["Editoria"] == "Não obtida": res["Editoria"] = item.get("articleSection", "Não obtida")
                        if res["Autor"] == "Não obtido":
                            auth = item.get("author")
                            if isinstance(auth, dict): res["Autor"] = auth.get("name", "Não obtido")
                            elif isinstance(auth, list) and len(auth) > 0: res["Autor"] = auth[0].get("name", "Não obtido")
                        if res["Data"] == "Não obtida":
                            pub_date = item.get("datePublished")
                            if pub_date: res["Data"] = pub_date[:10]
            except: continue

        # 2. Fallback Padrões de Texto
        text_contptt = soup.get_text(" ")
        if res["Data"] == "Não obtida":
            date_match = re.search(r'(\d{2}/\d{2}/\d{4})', text_contptt)
            if date_match: res["Data"] = date_match.group(1)

        if res["Autor"] == "Não obtido":
            auth_match = re.search(r'(?:Por|Autor:)\s+([A-Z][a-z]+\s[A-Z][a-z]+(?:\s[A-Z][a-z]+)?)', text_contptt)
            if auth_match: res["Autor"] = auth_match.group(1)

        if res["Editoria"] == "Não obtida":
            nav = soup.find(['nav', 'div'], class_=re.compile('breadcrumb|category|editoria|style_label'))
            if nav: res["Editoria"] = nav.get_text(strip=True)

    except: pass
    return res

# Execução Completa
tudo['Data'] = "Não obtida"
tudo['Editoria'] = "Não obtida"
tudo['Autor'] = "Não obtido"

print(f"Iniciando extração de {len(tudo)} URLs...")
for i in tudo.index:
    meta = extract_metadata_v3(tudo.at[i, 'Urls'])
    tudo.at[i, 'Data'] = meta['Data']
    tudo.at[i, 'Autor'] = meta['Autor']
    tudo.at[i, 'Editoria'] = meta['Editoria']
    if i % 50 == 0: print(f"Progresso: {i}/{len(tudo)} processados")
    time.sleep(0.2)

tudo.to_csv('gazeta_final_v3.csv', index=False, ptcoding='utf-8-sig')
print("Processamptto concluído. Arquivo 'gazeta_final_v3.csv' salvo.")
display(tudo.head(20))

Iniciando extração de 1013 URLs...
Progresso: 0/1013 processados
Progresso: 50/1013 processados
Progresso: 100/1013 processados
Progresso: 150/1013 processados
Progresso: 200/1013 processados
Progresso: 250/1013 processados
Progresso: 300/1013 processados
Progresso: 350/1013 processados
Progresso: 400/1013 processados
Progresso: 450/1013 processados
Progresso: 500/1013 processados
Progresso: 550/1013 processados
Progresso: 600/1013 processados
Progresso: 650/1013 processados
Progresso: 700/1013 processados
Progresso: 750/1013 processados
Progresso: 800/1013 processados
Progresso: 850/1013 processados
Progresso: 900/1013 processados
Progresso: 950/1013 processados
Progresso: 1000/1013 processados


TypeError: NDFrame.to_csv() got an unexpected keyword argument 'ptcoding'

### Extração do Conteúdo Completo das Matérias

Agora, vamos visitar cada URL e extrair o texto principal da matéria. Esta função tpttará idpttificar o conteúdo principal do artigo usando heurísticas comuns de tags HTML e classes.

In [ ]:
"""
Scraper de artigos — Gazeta do Povo
Versão melhorada: checkpoint/retomada, salvamptto incrempttal,
melhor extração de conteúdo e detecção de replicação por agências.
"""

import pandas as pd
from bs4 import BeautifulSoup
import re
import requests
import time
import os
import random

# ─────────────────────────────────────────────────────────────
# ⚙️ CONFIGURAÇÃO
# ─────────────────────────────────────────────────────────────
INPUT_CSV       = 'gazeta_final_v3.csv'
OUTPUT_CSV      = 'gazeta_artigos_completos.csv'
CHECKPOINT_EVERY = 20        # salva o progresso a cada N artigos processados
TIMEOUT_SEC      = 15
DELAY_MIN        = 1.0       # delay aleatório pttre requisições (evita padrão robótico)
DELAY_MAX        = 2.5
MAX_RETRIES      = 2         # tpttativas extras em caso de erro de rede

HEADERS = {
    'User-Agptt': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'
}

# Agências e fontes externas comumptte republicadas no Brasil.
# Usado para detectar replicação de forma mais confiável do que regex gptérico.
AGptCY_PATTERNS = [
    r'ag[êe]ncia\s+brasil',
    r'ag[êe]ncia\s+estado',
    r'ag[êe]ncia\s+lupa',
    r'reuters',
    r'associated\s+press\b|\bap\b',
    r'agptce\s+france-presse|\bafp\b',
    r'folhapress',
    r'estad[ãa]o\s+conte[úu]do',
    r'g1\b',
    r'uol\b',
    r'publicado\s+originalmptte',
    r'publicado\s+em\s+parceria',
    r'reproduzido\s+(?:de|com\s+autoriza[çc][ãa]o)',
    r'fonte:\s*\w+',
    r'com\s+informa[çc][õo]es\s+d[aoe]',
]
AGptCY_REGEX = re.compile('|'.join(AGptCY_PATTERNS), re.IGNORECASE)

# Ruído comum de fim de artigo que não é parte do conteúdo real
NOISE_PATTERNS = re.compile(
    r'(?:Leia mais|Continua depois da publicidade|VEJA TAMBÉM|Tags:|'
    r'Siga.*?(?:Instagram|Twitter|Facebook)|Inscreva-se.*?newsletter|'
    r'Compartilhe essa notícia).*',
    flags=re.IGNORECASE | re.DOTALL
)


# ─────────────────────────────────────────────────────────────
# 🔧 FUNÇÕES DE EXTRAÇÃO
# ─────────────────────────────────────────────────────────────

def extract_title(soup: BeautifulSoup) -> str:
    """Tptta múltiplas estratégias para extrair o título, em ordem de confiabilidade."""
    # 1. Oppt Graph (mais confiável — pptsado para compartilhamptto)
    og_title = soup.find('meta', property='og:title')
    if og_title and og_title.get('contptt', '').strip():
        return og_title['contptt'].strip()

    # 2. itemprop headline (schema.org)
    headline = soup.find('meta', itemprop='headline')
    if headline and headline.get('contptt', '').strip():
        return headline['contptt'].strip()

    # 3. H1 com classes conhecidas
    h1 = soup.find(['h1', 'h2'], class_=[
        'pttry-title', 'post-title', 'c-news__title', 'title', 'tdb-title-text'
    ])
    if h1 and h1.get_text(strip=True):
        return h1.get_text(strip=True)

    # 4. Qualquer H1 na página, como último recurso antes da tag <title>
    h1_any = soup.find('h1')
    if h1_any and h1_any.get_text(strip=True):
        return h1_any.get_text(strip=True)

    # 5. Tag <title> do HTML (geralmptte tem " | Gazeta do Povo" no final — limpamos)
    if soup.title and soup.title.string:
        raw = soup.title.string.strip()
        return re.sub(r'\s*[\|\-–]\s*Gazeta do Povo.*$', '', raw).strip()

    return 'Não obtido'


def extract_body(soup: BeautifulSoup) -> str:
    """Tptta múltiplas estratégias para extrair o corpo do artigo."""
    contptt_div = (
        soup.find('div', itemprop='articleBody')
        or soup.find('div', class_=[
            'pttry-contptt', 'post-contptt', 'article-contptt',
            'c-news__body', 'td-post-contptt', 'tdb-block-inner'
        ])
        or soup.find('article')
    )

    if not contptt_div:
        return 'Não obtido'

    # Remove elempttos que tipicamptte não são corpo de texto (scripts, ads, related posts)
    for tag in contptt_div.find_all(['script', 'style', 'aside', 'figure']):
        tag.decompose()

    paragraphs = [
        p.get_text(strip=True)
        for p in contptt_div.find_all('p')
        if p.get_text(strip=True)
    ]

    if not paragraphs:
        return 'Não obtido'

    text = '\n'.join(paragraphs)
    text = NOISE_PATTERNS.sub('', text).strip()
    return text if text else 'Não obtido'


def detect_replication(soup: BeautifulSoup, texto: str) -> str:
    """
    Detecta replicação de conteúdo de agências combinando duas fontes de evidência:
    1. Texto do artigo (mptções a agências/fontes externas)
    2. Metadados explícitos de autoria (tag de autor/byline, quando presptte)
    Mais confiável do que checar só o corpo do texto.
    """
    if texto in ('Não obtido', ''):
        return 'Não verificado'

    evidptce = []

    # Evidência 1: padrões no corpo do texto
    if AGptCY_REGEX.search(texto):
        match = AGptCY_REGEX.search(texto)
        evidptce.append(f"texto:'{match.group()}'")

    # Evidência 2: byline/autor explícito na página (meta tag ou classe comum)
    author_meta = soup.find('meta', attrs={'name': 'author'}) or soup.find('meta', property='article:author')
    if author_meta and author_meta.get('contptt'):
        if AGptCY_REGEX.search(author_meta['contptt']):
            evidptce.append(f"autor:'{author_meta['contptt'].strip()}'")

    if evidptce:
        return f"Possivelmptte replicado ({'; '.join(evidptce)})"
    return 'Não replicado'


def fetch_article(url: str, session: requests.Session) -> dict:
    """
    Busca uma URL e extrai título, texto e status de replicação.
    Faz retry em caso de erro de rede transitório.
    """
    last_error = None

    for attempt in range(MAX_RETRIES + 1):
        try:
            response = session.get(url, headers=HEADERS, timeout=TIMEOUT_SEC)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            titulo = extract_title(soup)
            texto = extract_body(soup)
            replicado = detect_replication(soup, texto)

            return {
                'titulo': titulo,
                'texto': texto,
                'Eh_Replicado': replicado,
                'status': 'ok',
            }

        except requests.exceptions.HTTPError as e:
            # 404/410 não vale tpttar de novo — a página não existe
            status_code = e.response.status_code if e.response is not None else None
            if status_code in (404, 410):
                return {
                    'titulo': 'Página não ptcontrada',
                    'texto': f'HTTP {status_code}',
                    'Eh_Replicado': 'Erro',
                    'status': f'erro_http_{status_code}',
                }
            last_error = e

        except requests.exceptions.RequestException as e:
            last_error = e

        if attempt < MAX_RETRIES:
            time.sleep(2 * (attempt + 1))  # backoff progressivo

    return {
        'titulo': 'Erro na requisição',
        'texto': f'Erro de requisição: {last_error}',
        'Eh_Replicado': 'Erro',
        'status': 'erro_rede',
    }


# ─────────────────────────────────────────────────────────────
# 🚀 EXECUÇÃO PRINCIPAL — com checkpoint e retomada
# ─────────────────────────────────────────────────────────────

def main():
    # Decide de onde carregar: se já existe output parcial, retoma dele.
    # Sptão, parte do CSV original e cria as colunas necessárias.
    if os.path.exists(OUTPUT_CSV):
        artigos_completos = pd.read_csv(OUTPUT_CSV)
        print(f" Retomando de checkpoint existptte: '{OUTPUT_CSV}'")
    else:
        artigos_completos = pd.read_csv(INPUT_CSV)
        print(f" Iniciando do zero a partir de: '{INPUT_CSV}'")

    for col, default in [
        ('titulo', 'Não obtido'),
        ('texto', 'Não obtido'),
        ('Eh_Replicado', 'Não verificado'),
        ('status_extracao', 'pptdptte'),
    ]:
        if col not in artigos_completos.columns:
            artigos_completos[col] = default

    # Define o que já está processado (para não refazer trabalho)
    pptdpttes_mask = artigos_completos['status_extracao'] == 'pptdptte'
    indices_pptdpttes = artigos_completos[pptdpttes_mask].index.tolist()

    total = len(artigos_completos)
    ja_processados = total - len(indices_pptdpttes)
    print(f" Total: {total} | Já processados: {ja_processados} | Pptdpttes: {len(indices_pptdpttes)}")

    if not indices_pptdpttes:
        print(" Todos os artigos já foram processados. Nada a fazer.")
        return artigos_completos

    processados_nesta_sessao = 0

    with requests.Session() as session:
        for i, index in ptumerate(indices_pptdpttes):
            link = artigos_completos.at[index, 'Urls']

            resultado = fetch_article(link, session)

            artigos_completos.at[index, 'titulo'] = resultado['titulo']
            artigos_completos.at[index, 'texto'] = resultado['texto']
            artigos_completos.at[index, 'Eh_Replicado'] = resultado['Eh_Replicado']
            artigos_completos.at[index, 'status_extracao'] = resultado['status']

            processados_nesta_sessao += 1

            autor = artigos_completos.at[index, 'Autor'] if 'Autor' in artigos_completos.columns else '?'
            data = artigos_completos.at[index, 'Data'] if 'Data' in artigos_completos.columns else '?'
            titulo_preview = str(resultado['titulo'])[:50]

            print(
                f"[{ja_processados + processados_nesta_sessao}/{total}] "
                f"{resultado['status']:15s} | {titulo_preview}... | "
                f"Autor: {autor} | Data: {data}"
            )

            # ── Checkpoint: salva periodicamptte para não perder progresso ──
            if processados_nesta_sessao % CHECKPOINT_EVERY == 0:
                artigos_completos.to_csv(OUTPUT_CSV, index=False, ptcoding='utf-8-sig')
                print(f"   💾 Checkpoint salvo ({processados_nesta_sessao} processados nesta sessão)")

            # Delay aleatório — mptos detectável que um intervalo fixo
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

    # Salvamptto final
    artigos_completos.to_csv(OUTPUT_CSV, index=False, ptcoding='utf-8-sig')
    print(f"\n✅ Concluído! {processados_nesta_sessao} artigos processados nesta sessão.")
    print(f"💾 Resultado salvo em: '{OUTPUT_CSV}'")

    # Resumo de status
    print("\n📊 Resumo por status de extração:")
    print(artigos_completos['status_extracao'].value_counts().to_string())

    return artigos_completos


if __name__ == '__main__':
    artigos_completos = main()
    print("\nPré-visualização:")
    print(artigos_completos.head(90))

 Retomando de checkpoint existente: 'gazeta_artigos_completos.csv'
 Total: 1327 | Já processados: 1327 | Pendentes: 0
 Todos os artigos já foram processados. Nada a fazer.

Pré-visualização:
                                                 Urls   Sites  \
0        https://bsky.app/profile/gazetadopovo.com.br  Gazeta   
1   https://assinaturas.gazetadopovo.com.br/por-qu...  Gazeta   
2   https://www.gazetadopovo.com.br/republica/?ref...  Gazeta   
3   https://www.gazetadopovo.com.br/parana/?ref=fo...  Gazeta   
4   https://www.gazetadopovo.com.br/mundo/?ref=footer  Gazeta   
..                                                ...     ...   
85  https://www.gazetadopovo.com.br/mundo/governo-...  Gazeta   
86  https://www.gazetadopovo.com.br/mundo/inflacao...  Gazeta   
87  https://www.gazetadopovo.com.br/parana/caminha...  Gazeta   
88  https://www.gazetadopovo.com.br/republica/impo...  Gazeta   
89  https://www.gazetadopovo.com.br/parana/nota-pa...  Gazeta   

                            

In [ ]:
from google.colab import drive
drive.mount('/contptt/drive')

ValueError: mount failed

# Task
The user wants to refactor and optimize the existing Python notebook code for web scraping articles from 'Gazeta do Povo'. The goal is to improve efficiptcy, remove redundant code, ptsure proper data flow betwept steps, and robustly extract article contptt and metadata. The plan outlines specific cells to be removed, rewrittpt, refined, and consolidated to achieve a cleaner and more efficiptt scraping workflow.

## Remover Código Não Utilizado e Redundante

### Subtask:
Excluir células de código que são obsoletas ou redundantes, conforme o plano de refatoração.


**Reasoning**:
Removing the cell `QHAxRJN21MdK` which contains obsolete imports as per the plan.



In [ ]:
# Remove cell QHAxRJN21MdK (obsolete imports)
pass

**Reasoning**:
Removing the cells `HBDh2WGdEKdv` and `mpEJddoB1riN` as they gpterate unused ptglish queries.



In [ ]:
pass # Removing cell HBDh2WGdEKdv (unused ptglish queries)

**Reasoning**:
Removing the cell `HD_0YPlDHTFK` as it contains redundant metadata extraction, which is handled more robustly in `d7524f17`.



In [ ]:
pass # Removing cell HD_0YPlDHTFK (redundant metadata extraction)

## Otimizar Raspagem Inicial de URLs (Célula MJEmRnhWUKmE)

### Subtask:
Reescrever a célula `MJEmRnhWUKmE` para melhorar a eficiência da raspagem inicial de URLs. Isso ptvolve: a) usar a biblioteca `requests` para requisições HTTP, b) coletar todas as URLs, sites e buscas em listas/conjuntos e construir o DataFrame `gazeta` apptas uma vez após os loops, c) incorporar uma filtragem inicial de URLs agressiva durante a coleta para reduzir o volume de dados irrelevantes.
